# Inference vs Inference Comparison: GS-GLS vs Baselines

This notebook performs a **fair performance comparison** by separating the "Training" phase (Covariance Estimation and Projection Matrix Computation) from the "Inference" phase (Matrix Multiplication) for the baseline methods (OLS, MinT).

Previously, baselines were re-calculating the projection matrix for every batch in the evaluation loop, leading to inflated "inference" times.

In [ ]:
import numpy as np
import pandas as pd
import time
import random
import matplotlib.pyplot as plt
import scipy.linalg
from scipy.linalg import inv, pinv

from hierarchy import Hierarchy
from data_generator import HierarchicalDataGenerator
from gs_gls import GSGLS
import baselines
from fair_baselines import BaselineReconciler

%load_ext autoreload
%autoreload 2

## 1. Setup Hierarchy and Data

In [ ]:
np.random.seed(42)

# 1. Define Random Hierarchy (Same as previous)
def generate_random_hierarchy(depth=4, branching_factor=5):
    structure = {}
    current_layer = ['Total']
    node_ctr = 1
    for d in range(depth):
        next_layer = []
        for parent in current_layer:
            n_children = random.randint(2, branching_factor)
            children = []
            for _ in range(n_children):
                child_name = f'Node_{d+1}_{node_ctr}'
                children.append(child_name)
                node_ctr += 1
            structure[parent] = children
            next_layer.extend(children)
        current_layer = next_layer
    return structure

structure = generate_random_hierarchy()
h = Hierarchy(structure)
print(f"Hierarchy: {h.n_nodes} nodes, {h.m_bottom} bottom nodes.")

# 2. Configuration
n_train = 500
n_test = 100
n_total = n_train + n_test
n_t_block = 10

gen = HierarchicalDataGenerator(h, n_timesteps=n_total)

def get_dataset(heteroscedastic=False):
    Y_true = gen.generate_ground_truth()
    # Generate residuals
    E = gen.generate_spatiotemporal_noise(
        spatial_rho=1.5,
        temporal_ar_coefs=[0.6, 0.2],
        noise_scale=2.0,
        heteroscedastic=heteroscedastic
    )
    Y_hat = Y_true + E
    
    # Split
    residuals_train = E[:, :n_train]
    Y_hat_test = Y_hat[:, n_train:]
    Y_true_test = Y_true[:, n_train:]
    
    return residuals_train, Y_hat_test, Y_true_test

# Precompute S matrix
S_sp = h.get_summing_matrix()
S_total = baselines.build_spatiotemporal_s(S_sp, n_t_block)

## 2. Evaluation Loop

In [ ]:
def run_comparison(scenario_name, residuals_train, Y_hat_test, Y_true_test):
    results = []
    
    # Prepare Residuals for Baselines (Flatten blocks)
    train_blocks = []
    for i in range(0, residuals_train.shape[1] - n_t_block + 1, n_t_block):
        block = residuals_train[:, i:i+n_t_block]
        train_blocks.append(block.flatten(order='F'))
    residuals_samples = np.array(train_blocks)
    
    # --- 1. Baselines ---
    baseline_methods = ['OLS', 'MinT_Sample', 'MinT_Shrink']
    
    for method_name in baseline_methods:
        # A. Training Time
        t0 = time.time()
        model = BaselineReconciler(method_name, S_total)
        model.fit(residuals_samples)
        train_time = time.time() - t0
        
        # B. Inference Time
        t0 = time.time()
        mse_list = []
        
        for i in range(0, Y_hat_test.shape[1] - n_t_block + 1, n_t_block):
            y_hat_blk = Y_hat_test[:, i:i+n_t_block]
            y_true_blk = Y_true_test[:, i:i+n_t_block]
            y_hat_flat = y_hat_blk.flatten(order='F')
            
            y_tilde_flat = model.reconcile(y_hat_flat)
            y_tilde_blk = y_tilde_flat.reshape((h.n_nodes, n_t_block), order='F')
            
            mse_list.append(np.mean((y_tilde_blk - y_true_blk)**2))
            
        infer_time = time.time() - t0
        
        results.append({
            'Method': method_name,
            'Train Time (s)': train_time,
            'Inference Time (s)': infer_time,
            'Total Time (s)': train_time + infer_time,
            'MSE': np.mean(mse_list)
        })
        
    # --- 2. GS-GLS ---
    gs_methods = ['spectral', 'wavelet']
    
    for tm in gs_methods:
        # A. Training Time
        t0 = time.time()
        gs = GSGLS(h, temporal_method=tm)
        gs.fit(residuals_train)
        train_time = time.time() - t0
        
        # B. Inference Time
        t0 = time.time()
        mse_list = []
        
        for i in range(0, Y_hat_test.shape[1] - n_t_block + 1, n_t_block):
            y_hat_blk = Y_hat_test[:, i:i+n_t_block]
            y_true_blk = Y_true_test[:, i:i+n_t_block]
            y_tilde_blk = gs.reconcile(y_hat_blk)
            
            mse_list.append(np.mean((y_tilde_blk - y_true_blk)**2))
            
        infer_time = time.time() - t0
        
        results.append({
            'Method': f"GS-GLS ({tm})",
            'Train Time (s)': train_time,
            'Inference Time (s)': infer_time,
            'Total Time (s)': train_time + infer_time,
            'MSE': np.mean(mse_list)
        })
        
    return pd.DataFrame(results)

## 4. Run Comparisons

In [ ]:
print("--- Scenario 1: Stationary Data ---")
res_train, Y_hat, Y_true = get_dataset(heteroscedastic=False)
df_res = run_comparison("Stationary", res_train, Y_hat, Y_true)
display(df_res)

print("\n--- Scenario 2: Non-Stationary Data ---")
res_train, Y_hat, Y_true = get_dataset(heteroscedastic=True)
df_res2 = run_comparison("Non-Stationary", res_train, Y_hat, Y_true)
display(df_res2)